# DS-6: Classification Algorithms Comparison
## Decision Tree vs Random Forest vs Support Vector Machine (SVM)

**Objective:** Train, tune, and compare three classifiers on the exact same dataset and train/test split.

**Dataset:** Breast Cancer Wisconsin Diagnostic dataset bundled with scikit-learn (569 rows, 30 numeric features; target classes are malignant and benign). No separate download is required.

**What this notebook covers**
- One shared stratified train/test split for all three models
- One tuned hyperparameter per model
- Accuracy, precision, recall, F1-score, confusion matrix, and training time
- Side-by-side comparison and a data-driven deployment recommendation

> Run the cells from top to bottom. Training time can vary by computer, so record the time printed by your own run.

## 1. Import libraries

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42


## 2. Load the dataset

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

# Make the class names explicit for easier interpretation:
# scikit-learn target 0 = malignant, target 1 = benign
print("Dataset shape:", X.shape)
print("\\nClass counts:")
print(y.map({0: "malignant", 1: "benign"}).value_counts())
print("\\nFeature preview:")
display(X.head())


## 3. Split the data ONCE
All three models use the same 80% training set and 20% test set. Stratification preserves the class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Train class proportions:\\n", y_train.value_counts(normalize=True).round(3))
print("Test class proportions:\\n", y_test.value_counts(normalize=True).round(3))


## 4. Define models and tune one hyperparameter per model

- **Decision Tree:** tune `max_depth` to limit tree complexity and reduce overfitting.
- **Random Forest:** tune `n_estimators`, the number of trees in the ensemble.
- **SVM:** tune `C` with an RBF kernel. Standardization is included in a pipeline because SVM is sensitive to feature scales.

Grid search uses cross-validation on the training set only. The held-out test set is kept for the final evaluation.

In [ ]:
models_and_grids = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {"max_depth": [3, 5, 8, None]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": [100, 200, 300]}
    ),
    "SVM (RBF)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf"))
        ]),
        {"svc__C": [0.1, 1, 10]}
    )
}

fitted_models = {}
results = []


## 5. Train, tune, and evaluate all models

In [ ]:
for model_name, (estimator, param_grid) in models_and_grids.items():
    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        refit=True
    )

    start = time.perf_counter()
    search.fit(X_train, y_train)
    training_seconds = time.perf_counter() - start

    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)

    # Treat malignant (class 0) as the positive class for safety-oriented recall.
    # This makes recall answer: "Of all malignant cases, how many did we identify?"
    metrics = {
        "Model": model_name,
        "Best hyperparameter": str(search.best_params_),
        "CV F1 (best)": round(search.best_score_, 4),
        "Test accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Test precision (malignant)": round(precision_score(y_test, y_pred, pos_label=0, zero_division=0), 4),
        "Test recall (malignant)": round(recall_score(y_test, y_pred, pos_label=0, zero_division=0), 4),
        "Test F1 (malignant)": round(f1_score(y_test, y_pred, pos_label=0, zero_division=0), 4),
        "Training time (seconds)": round(training_seconds, 4)
    }

    fitted_models[model_name] = best_model
    results.append(metrics)

    print(f"\\n===== {model_name} =====")
    print("Best parameters:", search.best_params_)
    print("Training + tuning time (seconds):", round(training_seconds, 4))
    print(classification_report(
        y_test, y_pred,
        labels=[0, 1],
        target_names=["malignant", "benign"],
        zero_division=0
    ))


## 6. Side-by-side comparison table

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by=["Test F1 (malignant)", "Test recall (malignant)", "Training time (seconds)"],
    ascending=[False, False, True]
).reset_index(drop=True)

display(results_df)


## 7. Confusion matrix for each model
Rows are actual classes; columns are predicted classes. Class 0 is malignant and class 1 is benign.

In [ ]:
for model_name, model in fitted_models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["malignant", "benign"]
    )
    disp.plot(values_format="d")
    plt.title(f"Confusion Matrix — {model_name}")
    plt.tight_layout()
    plt.show()


## 8. Interpretation and final recommendation

In [ ]:
# This creates a recommendation from the observed test results instead of hard-coding a winner.
# For this medical dataset, prioritize malignant-class recall, then F1, then training time.
ranked = results_df.sort_values(
    by=["Test recall (malignant)", "Test F1 (malignant)", "Training time (seconds)"],
    ascending=[False, False, True]
)
winner = ranked.iloc[0]

print("Observed top model under the stated priority:")
print(winner["Model"])
print("Malignant recall:", winner["Test recall (malignant)"])
print("Malignant F1:", winner["Test F1 (malignant)"])
print("Accuracy:", winner["Test accuracy"])
print("Training time (seconds):", winner["Training time (seconds)"])

print("\\nInterpretation guide:")
print("- Decision Tree: easiest to explain, but a deep tree can overfit.")
print("- Random Forest: averages many trees, usually more robust, but less interpretable and heavier.")
print("- SVM with RBF: can model nonlinear boundaries; scaling is important and tuning/training can be slower.")
print("- Do not select a model using accuracy alone. Here, malignant recall matters because missed malignant cases are costly.")
print("- This is an educational dataset and experiment, not a clinically validated diagnostic system.")


## 9. Assignment-ready written analysis (edit using your actual output)

**Aim:** To compare Decision Tree, Random Forest, and SVM classifiers on the same classification task using a common train/test split and consistent evaluation metrics.

**Methodology:** The Breast Cancer Wisconsin Diagnostic dataset was loaded from scikit-learn. The data was divided once into 80% training and 20% testing data using `random_state=42` and stratification. Each model was tuned with five-fold cross-validation on the training data. The Decision Tree's `max_depth`, Random Forest's `n_estimators`, and SVM's `C` were tuned. Standardization was applied inside the SVM pipeline to avoid data leakage. All models were evaluated on the same untouched test set.

**Metrics:** Accuracy measures overall correctness. Precision for the malignant class measures how many cases predicted malignant were actually malignant. Recall for the malignant class measures how many actual malignant cases were identified. F1-score balances precision and recall. The confusion matrix shows correct and incorrect predictions by class. Training time captures the cost of fitting and tuning on this machine.

**Discussion:** A Decision Tree is easy to interpret, but its performance can vary with tree depth. Random Forest combines many trees and often reduces the variance of a single tree, at the cost of interpretability and extra computation. SVM can create nonlinear boundaries with an RBF kernel, but it needs feature scaling and can become slower as datasets grow. The best choice depends on the observed recall, F1-score, confusion matrices, interpretability, and runtime—not accuracy alone.

**Conclusion:** Use the comparison table and confusion matrices generated above to name the strongest model for this run. For this dataset, give particular attention to malignant-class recall because false negatives are important. Report the actual metrics from your notebook; do not copy example numbers from another computer or run.

## 10. Optional: save the comparison table

In [ ]:
results_df.to_csv("classification_model_comparison.csv", index=False)
print("Saved: classification_model_comparison.csv")
